# Practice Day 13 · Aggregations and joins

`groupBy` with several aggregates, filters on aggregates, conditional counts, pivots, and every kind of join: anti, left, full, self, range and cross. In Zillacode's list these are the *Aggregate Functions*, *Complex Joins* and *Pivot Function* topics, the most common ones.

14 problems, about 2 hours. They prepare you for the Zillacode problems of Days 15 to 26.

## How to work
* Run the **Setup** cell first. `F` is `pyspark.sql.functions` and `Window` is `pyspark.sql.Window`, the same imports Zillacode's editor gives you (it calls the window class `W`).
* Every problem is in Zillacode's format: complete the function `etl(...)` in the cell under it, so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result.
* Your result must look exactly like the **expected output**: same columns, same rows, same order. Where an expected output is left-aligned, it was printed with `.show(truncate=False)`, as the cell already does.
* Stuck for 15 minutes? Open the hint under the problem. The answers are collapsed at the very end of the notebook: open one only after you have tried, then write the solution again yourself without looking.

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 "pandas>=2.2,<3" "pyarrow>=18" jupyterlab`, then `jupyter lab` and open this notebook.

## The data
The same small online shop as in the theory days. The setup cell creates every table, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is no longer in *products*.

This notebook also has:
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **customers_new**: the customers table one week later: some rows changed, one customer is gone, one is new

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("practice13")
         .config("spark.sql.shuffle.partitions", "4")          # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# promotions (a discount on one category for some days)
promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date, discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

# the customers table one week later
customers_new = spark.createDataFrame([
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
promotions.show()
customers_new.show()

## Problems

**Q1 · `etl(products)`** For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by category.

Expected output:

```
+-----------+--------+---------+---------+
|   category|products|avg_price|max_price|
+-----------+--------+---------+---------+
|      Books|       2|     20.0|     25.0|
|Electronics|       2|    675.0|   1200.0|
|       Home|       1|     40.0|     40.0|
|    Kitchen|       2|     72.5|     85.0|
+-----------+--------+---------+---------+
```

<details><summary>Hint</summary>

`groupBy(...).agg(...)` with several functions, each named with `.alias(...)`.

</details>

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q2 · `etl(orders)`** Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.

Expected output:

```
+-----------+------+
|customer_id|orders|
+-----------+------+
|          1|     4|
|          2|     3|
|          3|     3|
+-----------+------+
```

<details><summary>Hint</summary>

Aggregate, then `filter` on the new column (this is SQL's `HAVING`).

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q3 · `etl(orders, customers)`** How many different customers ordered in each month? Show month (1 to 3) and customers.

Expected output:

```
+-----+---------+
|month|customers|
+-----+---------+
|    1|        2|
|    2|        4|
|    3|        6|
+-----+---------+
```

<details><summary>Hint</summary>

`F.month(...)`, `F.count_distinct(...)`.

</details>

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q4 · `etl(orders)`** For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show customer_id, delivered and cancelled, sorted by customer_id. Leave out the guest order.

Expected output:

```
+-----------+---------+---------+
|customer_id|delivered|cancelled|
+-----------+---------+---------+
|          1|        3|        0|
|          2|        3|        0|
|          3|        2|        1|
|          4|        0|        1|
|          5|        1|        0|
|          6|        0|        0|
+-----------+---------+---------+
```

<details><summary>Hint</summary>

`F.sum(F.when(condition, 1).otherwise(0))` counts the rows where the condition is true.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q5 · `etl(orders)`** Make a table with one row per month and one column per status, holding the number of orders, with 0 where there are none. Columns: month, cancelled, delivered, shipped.

Expected output:

```
+-----+---------+---------+-------+
|month|cancelled|delivered|shipped|
+-----+---------+---------+-------+
|    1|        0|        3|      0|
|    2|        1|        2|      1|
|    3|        1|        5|      3|
+-----+---------+---------+-------+
```

<details><summary>Hint</summary>

`groupBy(...).pivot("status", [...]).count()` turns values into columns; `na.fill(0)` replaces the nulls.

</details>

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q6 · `etl(orders, products)`** What is the revenue of delivered orders in each category? Show category and revenue, highest first.

Expected output:

```
+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 2850.0|
|      Books|  205.0|
|    Kitchen|   60.0|
+-----------+-------+
```

<details><summary>Hint</summary>

Join first, then group. `F.sum` can add up an expression, not only a column.

</details>

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q7 · `etl(customers, orders)`** Which customers have never placed an order? Show customer_id and name.

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

<details><summary>Hint</summary>

The join type `"left_anti"` keeps the rows of the left table that have no match.

</details>

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q8 · `etl(customers, orders)`** Show every customer with their number of orders, including customers with 0 orders. Show name and orders, most orders first; on a tie, by name.

Expected output:

```
+----------------+------+
|            name|orders|
+----------------+------+
|    Aziz Karimov|     4|
|   Bella Schmidt|     3|
|     Carlos Ruiz|     3|
|Dilnoza Rahimova|     2|
|     Emma Wilson|     2|
|    Farid Aliyev|     1|
|       Grace Lee|     0|
+----------------+------+
```

<details><summary>Hint</summary>

A `left` join keeps customers without orders. Then count a column that is empty (null) for them.

</details>

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

**Q9 · `etl(orders, products)`** Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 when none). Products missing from the catalog show an empty name. Columns: product_id, product_name, orders.

Expected output:

```
+----------+------------+------+
|product_id|product_name|orders|
+----------+------------+------+
|       101|      Laptop|     3|
|       102|  Headphones|     2|
|       103|Coffee Maker|     3|
|       104|     Blender|     2|
|       105|       Novel|     3|
|       106|    Cookbook|     2|
|       107|   Desk Lamp|     0|
|       108|        NULL|     1|
+----------+------------+------+
```

<details><summary>Hint</summary>

A `"full"` join keeps the rows of both sides; then turn the missing counts into 0.

</details>

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

**Q10 · `etl(customers)`** Show the pairs of customers who live in the same city: city, customer_a and customer_b (names; each pair once, customer_a with the smaller customer_id).

Expected output:

```
+--------+-------------+----------------+
|    city|   customer_a|      customer_b|
+--------+-------------+----------------+
|  Berlin|Bella Schmidt|       Grace Lee|
|Tashkent| Aziz Karimov|Dilnoza Rahimova|
+--------+-------------+----------------+
```

<details><summary>Hint</summary>

A self join: the table joined with itself. `a.customer_id < b.customer_id` keeps each pair once and drops pairs of a customer with themself.

</details>

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q11 · `etl(orders, products, promotions)`** For each delivered order, find the promotion of its product's category that was running on the order date. Show order_id, product_name, promo_id and discount_pct; orders without a promotion drop out.

Expected output:

```
+--------+------------+--------+------------+
|order_id|product_name|promo_id|discount_pct|
+--------+------------+--------+------------+
|    1001|      Laptop|  WINTER|          10|
|    1003|  Headphones|  WINTER|          10|
|    1009|       Novel|BOOKWEEK|          20|
|    1011|     Blender|  SPRING|          15|
|    1013|      Laptop|   FLASH|           5|
+--------+------------+--------+------------+
```

<details><summary>Hint</summary>

A range join: the join condition is an expression, `category` equal and `order_date` between start_date and end_date.

</details>

In [ ]:
def etl(orders, products, promotions):
    # Write code here: return a DataFrame
    return orders

etl(orders, products, promotions).show()

**Q12 · `etl(customers, products, orders)`** Make a table with every combination of customers 1 to 3 and every product category, with the number of orders (0 when none). Columns: customer_id, category, orders.

Expected output:

```
+-----------+-----------+------+
|customer_id|   category|orders|
+-----------+-----------+------+
|          1|      Books|     1|
|          1|Electronics|     2|
|          1|       Home|     0|
|          1|    Kitchen|     1|
|          2|      Books|     1|
|          2|Electronics|     2|
|          2|       Home|     0|
|          2|    Kitchen|     0|
|          3|      Books|     1|
|          3|Electronics|     0|
|          3|       Home|     0|
|          3|    Kitchen|     1|
+-----------+-----------+------+
```

<details><summary>Hint</summary>

`crossJoin` builds all combinations; then left-join the real counts on both columns.

</details>

In [ ]:
def etl(customers, products, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, products, orders).show()

**Q13 · `etl(orders, customers)`** Which customers had all of their orders delivered (and at least one order)? Show their names.

Expected output:

```
+-------------+
|         name|
+-------------+
|Bella Schmidt|
+-------------+
```

<details><summary>Hint</summary>

Customers with orders (`left_semi`) minus customers with a non-delivered order (`left_anti`).

</details>

In [ ]:
def etl(orders, customers):
    # Write code here: return a DataFrame
    return orders

etl(orders, customers).show()

**Q14 · `etl(customers, customers_new)`** Compare `customers` with `customers_new`: which customers were `added`, `removed` or `changed` (a different name, email or city)? Show customer_id and change.

Expected output:

```
+-----------+-------+
|customer_id| change|
+-----------+-------+
|          2|changed|
|          6|changed|
|          7|removed|
|          8|  added|
+-----------+-------+
```

<details><summary>Hint</summary>

A full outer join on customer_id. Compare with `eqNullSafe`: with `==`, null compared to anything is null, so a missing email that got a value would not count as changed.

</details>

In [ ]:
def etl(customers, customers_new):
    # Write code here: return a DataFrame
    return customers

etl(customers, customers_new).show()

## Answers
Try each problem yourself first. Click a problem to open its answer.

<details>
<summary><b>Q1</b> &nbsp;For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by ...</summary>

```python
def etl(products):
    return (products.groupBy("category")
                    .agg(F.count("*").alias("products"),
                         F.round(F.avg("price"), 2).alias("avg_price"),
                         F.max("price").alias("max_price"))
                    .orderBy("category"))

etl(products).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.</summary>

```python
def etl(orders):
    return (orders.groupBy("customer_id")
                  .agg(F.count("*").alias("orders"))
                  .filter(F.col("orders") >= 3)
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;How many different customers ordered in each month? Show month (1 to 3) and customers.</summary>

```python
def etl(orders, customers):
    return (orders.groupBy(F.month("order_date").alias("month"))
                  .agg(F.count_distinct("customer_id").alias("customers"))
                  .orderBy("month"))

etl(orders, customers).show()
```

`count_distinct` skips nulls, so the guest order (no customer_id) does not count in March.

</details>

<details>
<summary><b>Q4</b> &nbsp;For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show ...</summary>

```python
def etl(orders):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
                       F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Make a table with one row per month and one column per status, holding the number of orders, with 0 where ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.month("order_date").alias("month"))
                  .pivot("status", ["cancelled", "delivered", "shipped"])
                  .count()
                  .na.fill(0)
                  .orderBy("month"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;What is the revenue of delivered orders in each category? Show category and revenue, highest first.</summary>

```python
def etl(orders, products):
    return (orders.filter(F.col("status") == "delivered")
                  .join(products, "product_id")
                  .groupBy("category")
                  .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue"))
                  .orderBy(F.col("revenue").desc()))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Which customers have never placed an order? Show customer_id and name.</summary>

```python
def etl(customers, orders):
    return customers.join(orders, "customer_id", "left_anti").select("customer_id", "name")

etl(customers, orders).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;Show every customer with their number of orders, including customers with 0 orders. Show name and orders, ...</summary>

```python
def etl(customers, orders):
    return (customers.join(orders, "customer_id", "left")
                     .groupBy("customer_id", "name")
                     .agg(F.count("order_id").alias("orders"))
                     .select("name", "orders")
                     .orderBy(F.col("orders").desc(), "name"))

etl(customers, orders).show()
```

`F.count("order_id")` skips nulls, so Grace gets 0. `F.count("*")` would give her 1: her joined row exists, it just has empty order columns.

</details>

<details>
<summary><b>Q9</b> &nbsp;Show every product of the catalog and every product_id that appears in orders, with its number of orders (0 ...</summary>

```python
def etl(orders, products):
    counts = orders.groupBy("product_id").agg(F.count("*").alias("orders"))
    return (products.join(counts, "product_id", "full")
                    .select("product_id", "product_name", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
                    .orderBy("product_id"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;Show the pairs of customers who live in the same city: city, customer_a and customer_b (names; each pair ...</summary>

```python
def etl(customers):
    a, b = customers.alias("a"), customers.alias("b")
    return (a.join(b, (F.col("a.city") == F.col("b.city")) & (F.col("a.customer_id") < F.col("b.customer_id")))
             .select(F.col("a.city").alias("city"), F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
             .orderBy("city"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q11</b> &nbsp;For each delivered order, find the promotion of its product's category that was running on the order date. ...</summary>

```python
def etl(orders, products, promotions):
    return (orders.filter(F.col("status") == "delivered")
                  .join(products, "product_id")
                  .join(promotions, (products.category == promotions.category)
                                    & F.col("order_date").between(F.col("start_date"), F.col("end_date")))
                  .select("order_id", "product_name", "promo_id", "discount_pct")
                  .orderBy("order_id"))

etl(orders, products, promotions).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;Make a table with every combination of customers 1 to 3 and every product category, with the number of orders ...</summary>

```python
def etl(customers, products, orders):
    combos = customers.filter(F.col("customer_id") <= 3).select("customer_id").crossJoin(products.select("category").distinct())
    counts = orders.join(products, "product_id").groupBy("customer_id", "category").agg(F.count("*").alias("orders"))
    return (combos.join(counts, ["customer_id", "category"], "left")
                  .select("customer_id", "category", F.coalesce(F.col("orders"), F.lit(0)).alias("orders"))
                  .orderBy("customer_id", "category"))

etl(customers, products, orders).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Which customers had all of their orders delivered (and at least one order)? Show their names.</summary>

```python
def etl(orders, customers):
    not_delivered = orders.filter(F.col("status") != "delivered").select("customer_id")
    return (customers.join(orders, "customer_id", "left_semi")
                     .join(not_delivered, "customer_id", "left_anti")
                     .select("name")
                     .orderBy("name"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;Compare customers with customers_new: which customers were added, removed or changed (a different name, email ...</summary>

```python
def etl(customers, customers_new):
    old, new = customers.alias("old"), customers_new.alias("new")
    same = (F.col("old.name").eqNullSafe(F.col("new.name"))
            & F.col("old.email").eqNullSafe(F.col("new.email"))
            & F.col("old.city").eqNullSafe(F.col("new.city")))
    return (old.join(new, F.col("old.customer_id") == F.col("new.customer_id"), "full")
               .select(F.coalesce(F.col("old.customer_id"), F.col("new.customer_id")).alias("customer_id"),
                       F.when(F.col("old.customer_id").isNull(), "added")
                        .when(F.col("new.customer_id").isNull(), "removed")
                        .when(~same, "changed").alias("change"))
               .filter(F.col("change").isNotNull())
               .orderBy("customer_id"))

etl(customers, customers_new).show()
```

</details>